In [ ]:
from pathlib import Path
from datetime import datetime
import sys

from pyspark.sql import functions as F



# --------------------------------------------------
# Project setup
# --------------------------------------------------

current_path = Path.cwd()

project_root = next(
    path
    for path in [current_path, *current_path.parents]
    if (path / "backend").exists()
)

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


# --------------------------------------------------
# Project imports
# --------------------------------------------------

from backend.src.ingestion.spark_session import create_spark_session
from backend.src.ingestion.load_trip_data import load_trip_data
from backend.src.processing.clean_trip_data import clean_trip_data


print(f"Project root: {project_root}")

Project root: c:\Dev\Projects\NycTaxiDemandForecasting


In [12]:
spark = create_spark_session()

raw_data_path = (
    project_root
    / "data"
    / "raw"
)

trips = load_trip_data(
    spark=spark,
    raw_data_path=raw_data_path,
)

print(f"Raw records: {trips.count():,}")

Found 6 parquet files:
  c:\Dev\Projects\NycTaxiDemandForecasting\data\raw\yellow_tripdata_2025-01.parquet
  c:\Dev\Projects\NycTaxiDemandForecasting\data\raw\yellow_tripdata_2025-02.parquet
  c:\Dev\Projects\NycTaxiDemandForecasting\data\raw\yellow_tripdata_2025-03.parquet
  c:\Dev\Projects\NycTaxiDemandForecasting\data\raw\yellow_tripdata_2025-04.parquet
  c:\Dev\Projects\NycTaxiDemandForecasting\data\raw\yellow_tripdata_2025-05.parquet
  c:\Dev\Projects\NycTaxiDemandForecasting\data\raw\yellow_tripdata_2025-06.parquet
Raw records: 24,083,384


## 3. Apply Existing Cleaning Logic

Before performing business-specific analysis, the project's existing
trip-cleaning pipeline is applied.

The current cleaning process:

- restricts observations to January–June 2025,
- removes trips where drop-off occurs before pickup,
- identifies and removes matching fare correction/reversal records.

This provides a consistent starting point with the demand forecasting
pipeline. Additional business-specific validation will be performed
later for monetary values and trip distances.

In [13]:
cleaned_trips = clean_trip_data(
    trips,
    start_date=datetime(2025, 1, 1),
    end_date=datetime(2025, 7, 1),
)

raw_count = trips.count()
cleaned_count = cleaned_trips.count()

removed_count = raw_count - cleaned_count
removed_pct = (
    removed_count / raw_count * 100
)

print(f"Raw records:     {raw_count:,}")
print(f"Cleaned records: {cleaned_count:,}")
print(f"Removed records: {removed_count:,}")
print(f"Removed share:   {removed_pct:.2f}%")

Raw records:     24,083,384
Cleaned records: 23,359,857
Removed records: 723,527
Removed share:   3.00%


## 4. Business Data Quality Assessment

The general cleaning pipeline leaves approximately 23.36 million valid trip
records for January–June 2025.

However, a dataset that is suitable for demand forecasting is not necessarily
ready for financial analysis. Business metrics such as fare revenue, passenger
charges, tips, and fare per mile are particularly sensitive to negative values,
zero-distance trips, and extreme outliers.

This section therefore examines the quality and distribution of the main
business variables before defining any additional cleaning rules.

In [14]:
business_metrics = [
    "fare_amount",
    "total_amount",
    "tip_amount",
    "trip_distance",
]

cleaned_trips.select(
    *business_metrics
).summary(
    "count",
    "min",
    "mean",
    "max",
).show(truncate=False)

+-------+------------------+------------------+------------------+-----------------+
|summary|fare_amount       |total_amount      |tip_amount        |trip_distance    |
+-------+------------------+------------------+------------------+-----------------+
|count  |23359857          |23359857          |23359857          |23359857         |
|min    |-801.0            |-806.75           |-70.07            |0.0              |
|mean   |18.439165189239656|27.175084806813928|2.9423023689735555|6.917233197530851|
|max    |863372.12         |863380.37         |960.94            |386088.43        |
+-------+------------------+------------------+------------------+-----------------+



In [15]:
quality_checks = cleaned_trips.select(
    F.sum(
        (F.col("fare_amount") < 0).cast("long")
    ).alias("negative_fares"),

    F.sum(
        (F.col("total_amount") < 0).cast("long")
    ).alias("negative_total_amounts"),

    F.sum(
        (F.col("tip_amount") < 0).cast("long")
    ).alias("negative_tips"),

    F.sum(
        (F.col("trip_distance") == 0).cast("long")
    ).alias("zero_distance_trips"),

    F.sum(
        (F.col("trip_distance") < 0).cast("long")
    ).alias("negative_distance_trips"),
)

quality_checks.show(truncate=False)

+--------------+----------------------+-------------+-------------------+-----------------------+
|negative_fares|negative_total_amounts|negative_tips|zero_distance_trips|negative_distance_trips|
+--------------+----------------------+-------------+-------------------+-----------------------+
|961693        |84956                 |80           |610312             |0                      |
+--------------+----------------------+-------------+-------------------+-----------------------+



### Distribution and Outlier Analysis

The initial quality checks reveal a substantial number of negative fare
amounts and zero-distance trips. These observations are not removed
automatically because unusual values may represent legitimate fare structures,
adjustments, corrections, or special trip cases.

Quantiles are therefore examined to distinguish the normal range of the data
from extreme observations before defining business-specific cleaning rules.

In [16]:
columns = [
    "fare_amount",
    "total_amount",
    "tip_amount",
    "trip_distance",
]

probabilities = [
    0.50,
    0.90,
    0.95,
    0.99,
    0.995,
    0.999,
]

for column in columns:
    quantiles = cleaned_trips.approxQuantile(
        column,
        probabilities,
        0.0001,
    )

    print(f"\n--- {column} ---")

    for probability, value in zip(
        probabilities,
        quantiles,
    ):
        print(
            f"{probability * 100:>5.1f}%: "
            f"{value:,.2f}"
        )


--- fare_amount ---
 50.0%: 13.50
 90.0%: 37.50
 95.0%: 54.80
 99.0%: 74.40
 99.5%: 88.00
 99.9%: 135.30

--- total_amount ---
 50.0%: 21.06
 90.0%: 48.80
 95.0%: 75.75
 99.0%: 103.86
 99.5%: 112.85
 99.9%: 168.50

--- tip_amount ---
 50.0%: 2.30
 90.0%: 6.45
 95.0%: 10.45
 99.0%: 17.34
 99.5%: 20.25
 99.9%: 28.10

--- trip_distance ---
 50.0%: 1.80
 90.0%: 8.18
 95.0%: 12.00
 99.0%: 19.33
 99.5%: 21.00
 99.9%: 28.80


### Investigation of Negative Fare Amounts

Negative fare amounts occur frequently enough that they should not be removed
without further investigation.

This section examines whether negative fares are associated with particular
payment types, rate codes, total charges, or trip characteristics. The goal is
to distinguish meaningful adjustment or reversal records from observations
that should be excluded from business revenue calculations.

In [17]:
negative_fares = cleaned_trips.filter(
    F.col("fare_amount") < 0
)

negative_fares.select(
    "fare_amount",
    "total_amount",
    "tip_amount",
    "trip_distance",
    "payment_type",
    "RatecodeID",
).summary(
    "count",
    "min",
    "mean",
    "max",
).show(truncate=False)

+-------+-------------------+------------------+---------------------+------------------+-------------------+------------------+
|summary|fare_amount        |total_amount      |tip_amount           |trip_distance     |payment_type       |RatecodeID        |
+-------+-------------------+------------------+---------------------+------------------+-------------------+------------------+
|count  |961693             |961693            |961693               |961693            |961693             |44341             |
|min    |-801.0             |-806.75           |-70.07               |0.0               |0                  |1                 |
|mean   |-5.1269193079288735|2.7101456493912197|0.0028959865570405527|19.705409117046994|0.15683591333200927|1.1831487787826165|
|max    |-0.01              |101.53            |80.7                 |320136.29         |4                  |5                 |
+-------+-------------------+------------------+---------------------+------------------+--------

In [18]:
(
    negative_fares
    .groupBy("payment_type")
    .agg(
        F.count("*").alias("trip_count"),
        F.avg("fare_amount").alias("avg_fare"),
        F.avg("total_amount").alias("avg_total"),
    )
    .orderBy(
        F.desc("trip_count")
    )
    .show(truncate=False)
)

+------------+----------+-------------------+-------------------+
|payment_type|trip_count|avg_fare           |avg_total          |
+------------+----------+-------------------+-------------------+
|0           |917352    |-4.390970892307465 |4.098537584264273  |
|4           |28511     |-20.316040475605906|-25.97952965522079 |
|2           |10656     |-20.922955142642643|-26.652407094594597|
|3           |5149      |-19.377090697222755|-24.85951835307826 |
|1           |25        |-19.927999999999997|-30.458799999999997|
+------------+----------+-------------------+-------------------+



In [19]:
(
    negative_fares
    .withColumn(
        "ratecode_missing",
        F.col("RatecodeID").isNull(),
    )
    .groupBy(
        "payment_type",
        "ratecode_missing",
    )
    .agg(
        F.count("*").alias("trip_count"),
        F.avg("fare_amount").alias("avg_fare"),
        F.avg("total_amount").alias("avg_total"),
        F.avg("trip_distance").alias("avg_distance"),
    )
    .orderBy(
        "payment_type",
        F.desc("trip_count"),
    )
    .show(truncate=False)
)

+------------+----------------+----------+-------------------+-------------------+------------------+
|payment_type|ratecode_missing|trip_count|avg_fare           |avg_total          |avg_distance      |
+------------+----------------+----------+-------------------+-------------------+------------------+
|0           |true            |917352    |-4.390970892307465 |4.098537584264273  |20.363744211600594|
|1           |false           |25        |-19.927999999999997|-30.458799999999997|3.372             |
|2           |false           |10656     |-20.922955142642643|-26.652407094594597|3.3674230480480483|
|3           |false           |5149      |-19.377090697222755|-24.85951835307826 |2.7539250339871812|
|4           |false           |28511     |-20.316040475605906|-25.97952965522079 |7.7052720002805986|
+------------+----------------+----------+-------------------+-------------------+------------------+



In [20]:
(
    negative_fares
    .filter(
        F.col("payment_type") == 0
    )
    .withColumn(
        "month",
        F.month("tpep_pickup_datetime"),
    )
    .groupBy("month")
    .agg(
        F.count("*").alias("trip_count"),
        F.avg("fare_amount").alias("avg_fare"),
        F.avg("total_amount").alias("avg_total"),
        F.avg("trip_distance").alias("avg_distance"),
    )
    .orderBy("month")
    .show(truncate=False)
)

+-----+----------+-------------------+------------------+------------------+
|month|trip_count|avg_fare           |avg_total         |avg_distance      |
+-----+----------+-------------------+------------------+------------------+
|1    |84822     |-4.5745180495626085|3.585951050435031 |25.826796821579315|
|2    |127878    |-4.352346846212796 |4.122891505966629 |16.00704030403982 |
|3    |140460    |-4.266466467321659 |4.508869642602872 |25.887583724903934|
|4    |114421    |-4.315062095244744 |4.3438758619484235|21.994311446325437|
|5    |247658    |-4.402423463001409 |3.375452519199857 |18.77243206357159 |
|6    |202113    |-4.45384393878675  |4.760221856090391 |18.01550875005573 |
+-----+----------+-------------------+------------------+------------------+



In [21]:
(
    negative_fares
    .filter(F.col("payment_type") == 0)
    .select(
        "tpep_pickup_datetime",
        "PULocationID",
        "DOLocationID",
        "trip_distance",
        "fare_amount",
        "extra",
        "mta_tax",
        "tip_amount",
        "tolls_amount",
        "improvement_surcharge",
        "congestion_surcharge",
        "Airport_fee",
        "cbd_congestion_fee",
        "total_amount",
        "RatecodeID",
        "payment_type",
    )
    .show(20, truncate=False)
)

+--------------------+------------+------------+-------------+-----------+-----+-------+----------+------------+---------------------+--------------------+-----------+------------------+------------+----------+------------+
|tpep_pickup_datetime|PULocationID|DOLocationID|trip_distance|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|congestion_surcharge|Airport_fee|cbd_congestion_fee|total_amount|RatecodeID|payment_type|
+--------------------+------------+------------+-------------+-----------+-----+-------+----------+------------+---------------------+--------------------+-----------+------------------+------------+----------+------------+
|2025-01-01 00:14:52 |234         |237         |2.09         |-3.88      |0.0  |0.5    |0.0       |0.0         |1.0                  |NULL                |NULL       |0.0               |0.12        |NULL      |0           |
|2025-01-01 00:27:37 |49          |75          |12.28        |-9.69      |0.0  |0.5    |0.0       |0.0  

In [22]:
flex_fares = (
    cleaned_trips
    .filter(F.col("payment_type") == 0)
)

flex_fares = flex_fares.withColumn(
    "calculated_total",
    F.coalesce(F.col("fare_amount"), F.lit(0.0))
    + F.coalesce(F.col("extra"), F.lit(0.0))
    + F.coalesce(F.col("mta_tax"), F.lit(0.0))
    + F.coalesce(F.col("tip_amount"), F.lit(0.0))
    + F.coalesce(F.col("tolls_amount"), F.lit(0.0))
    + F.coalesce(F.col("improvement_surcharge"), F.lit(0.0))
    + F.coalesce(F.col("congestion_surcharge"), F.lit(0.0))
    + F.coalesce(F.col("Airport_fee"), F.lit(0.0))
    + F.coalesce(F.col("cbd_congestion_fee"), F.lit(0.0))
)

flex_fares = flex_fares.withColumn(
    "total_difference",
    F.col("total_amount") - F.col("calculated_total")
)

flex_fares.select(
    F.count("*").alias("records"),
    F.avg("total_difference").alias("avg_difference"),
    F.min("total_difference").alias("min_difference"),
    F.max("total_difference").alias("max_difference"),
    F.avg(F.abs("total_difference")).alias("avg_abs_difference"),
).show(truncate=False)

+-------+-----------------+-------------------+--------------+------------------+
|records|avg_difference   |min_difference     |max_difference|avg_abs_difference|
+-------+-----------------+-------------------+--------------+------------------+
|5417852|2.728513220737657|-13.540000000000001|227.45        |2.72890945895162  |
+-------+-----------------+-------------------+--------------+------------------+



In [23]:
(
    flex_fares
    .select(
        "fare_amount",
        "calculated_total",
        "total_amount",
        "total_difference",
        "trip_distance",
    )
    .orderBy(
        F.desc(F.abs("total_difference"))
    )
    .show(20, truncate=False)
)

+-----------+---------------------+------------+-----------------+-------------+
|fare_amount|calculated_total     |total_amount|total_difference |trip_distance|
+-----------+---------------------+------------+-----------------+-------------+
|0.0        |15.610000000000001   |243.06      |227.45           |79.73        |
|-4.75      |-2.5                 |100.0       |102.5            |4.94         |
|-4.75      |-2.5                 |100.0       |102.5            |3.79         |
|-4.75      |-2.5                 |100.0       |102.5            |0.02         |
|-4.75      |-2.5                 |100.0       |102.5            |4.85         |
|-4.93      |4.440892098500626E-16|101.53      |101.53           |211.55       |
|-11.69     |-2.499999999999999   |85.6        |88.1             |10.55        |
|2.9        |10.64                |97.27       |86.63            |22.1         |
|0.0        |7.74                 |91.27       |83.53            |32.74        |
|2.9        |3.6999999999999

In [25]:
payment_type_0_trips = flex_fares

In [26]:
distance_analysis = (
    payment_type_0_trips
    .filter(
        (F.col("trip_distance") >= 0)
        & (F.col("trip_distance") <= 30)
    )
    .withColumn(
        "distance_bin",
        F.floor("trip_distance"),
    )
    .groupBy("distance_bin")
    .agg(
        F.count("*").alias("trip_count"),
        F.avg("total_difference").alias(
            "avg_total_difference"
        ),
        F.avg("fare_amount").alias(
            "avg_fare_amount"
        ),
        F.avg("total_amount").alias(
            "avg_total_amount"
        ),
    )
    .orderBy("distance_bin")
)

distance_analysis_pd = (
    distance_analysis.toPandas()
)

In [28]:
import pandas as pd
import plotly.express as px

In [29]:
fig = px.line(
    distance_analysis_pd,
    x="distance_bin",
    y="avg_total_difference",
    markers=True,
    title=(
        "Average Total Amount Difference "
        "by Trip Distance"
    ),
    labels={
        "distance_bin": "Trip Distance (miles)",
        "avg_total_difference":
            "Average Total Difference ($)",
    },
)

fig.show()

In [30]:
fig = px.bar(
    distance_analysis_pd,
    x="distance_bin",
    y="trip_count",
    title="Trip Volume by Distance",
    labels={
        "distance_bin": "Trip Distance (miles)",
        "trip_count": "Number of Trips",
    },
)

fig.show()

In [31]:
fig = px.scatter(
    distance_analysis_pd,
    x="distance_bin",
    y="avg_total_difference",
    size="trip_count",
    hover_data=[
        "trip_count",
        "avg_fare_amount",
        "avg_total_amount",
    ],
    title=(
        "Average Total Amount Difference "
        "by Trip Distance and Volume"
    ),
    labels={
        "distance_bin": "Trip Distance (miles)",
        "avg_total_difference":
            "Average Total Difference ($)",
        "trip_count": "Number of Trips",
    },
)

fig.show()

In [32]:
payment_type_analysis = (
    cleaned_trips
    .groupBy("payment_type")
    .agg(
        F.count("*").alias("trip_count"),
        F.avg("fare_amount").alias("avg_fare"),
        F.avg("total_amount").alias("avg_total"),
        F.avg("tip_amount").alias("avg_tip"),
        F.avg("trip_distance").alias("avg_distance"),
        F.sum("total_amount").alias("total_amount_sum"),
    )
    .orderBy("payment_type")
)

payment_type_analysis_pd = (
    payment_type_analysis.toPandas()
)

payment_type_analysis_pd.round(2)

,payment_type,trip_count,avg_fare,avg_total,avg_tip,avg_distance,total_amount_sum
0,0,5417852,16.59,21.97,0.34,18.16,1.190234e+08
1,1,15608319,19.10,29.49,4.28,3.55,4.603647e+08
2,2,2176826,18.52,24.12,0.00,3.23,5.250735e+07
3,3,65131,14.77,19.23,0.01,2.31,1.252146e+06
4,4,91726,16.00,18.08,0.02,5.83,1.658446e+06
5,5,3,17.27,24.00,0.00,3.33,7.199000e+01


In [33]:
fig = px.bar(
    payment_type_analysis_pd,
    x="payment_type",
    y="trip_count",
    title="Trip Volume by Payment Type",
    labels={
        "payment_type": "Payment Type",
        "trip_count": "Number of Trips",
    },
)

fig.show()

## Business-Oriented Data Cleaning

The technical cleaning stage preserves records that may still be valid from a
data-engineering perspective. Revenue analysis, however, requires additional
business rules.

Exploratory analysis identified a distinct group with `payment_type = 0`.
These trips exhibit a different fare structure, frequently missing `RatecodeID`
values and systematic differences between the reported `total_amount` and the
sum of standard fare components. They are therefore retained in the source
dataset but excluded from the initial standard revenue analysis.

The business dataset also excludes negative monetary values and extreme
positive observations that would distort aggregate revenue metrics.

Zero-distance trips are not removed globally because they may still represent
valid charged transactions. They can be excluded separately when analysing
relationships involving trip distance.

In [34]:
business_trips = (
    cleaned_trips
    .filter(
        (F.col("payment_type").isin(1, 2, 3, 4, 5))
        & (F.col("fare_amount") >= 0)
        & (F.col("total_amount") >= 0)
    )
)

In [35]:
cleaned_count = cleaned_trips.count()
business_count = business_trips.count()

excluded_count = (
    cleaned_count - business_count
)

excluded_share = (
    excluded_count
    / cleaned_count
    * 100
)

print(f"Cleaned trips:   {cleaned_count:,}")
print(f"Business trips:  {business_count:,}")
print(f"Excluded trips:  {excluded_count:,}")
print(f"Excluded share:  {excluded_share:.2f}%")

Cleaned trips:   23,359,857
Business trips:  17,895,220
Excluded trips:  5,464,637
Excluded share:  23.39%


In [36]:
cleaning_audit = cleaned_trips.agg(
    F.count("*").alias("cleaned_trips"),

    F.sum(
        F.when(
            F.col("payment_type") == 0,
            1,
        ).otherwise(0)
    ).alias("payment_type_0"),

    F.sum(
        F.when(
            F.col("fare_amount") < 0,
            1,
        ).otherwise(0)
    ).alias("negative_fare"),

    F.sum(
        F.when(
            F.col("total_amount") < 0,
            1,
        ).otherwise(0)
    ).alias("negative_total"),

    F.sum(
        F.when(
            F.col("payment_type").isNull(),
            1,
        ).otherwise(0)
    ).alias("missing_payment_type"),

    F.sum(
        F.when(
            ~F.col("payment_type").isin(0, 1, 2, 3, 4, 5),
            1,
        ).otherwise(0)
    ).alias("other_payment_type"),
)

cleaning_audit.show(truncate=False)

+-------------+--------------+-------------+--------------+--------------------+------------------+
|cleaned_trips|payment_type_0|negative_fare|negative_total|missing_payment_type|other_payment_type|
+-------------+--------------+-------------+--------------+--------------------+------------------+
|23359857     |5417852       |961693       |84956         |0                   |0                 |
+-------------+--------------+-------------+--------------+--------------------+------------------+



In [37]:
cleaning_breakdown = (
    cleaned_trips
    .withColumn(
        "business_status",
        F.when(
            F.col("payment_type") == 0,
            "Excluded: payment_type 0",
        )
        .when(
            F.col("payment_type").isNull(),
            "Excluded: missing payment type",
        )
        .when(
            F.col("fare_amount") < 0,
            "Excluded: negative fare",
        )
        .when(
            F.col("total_amount") < 0,
            "Excluded: negative total",
        )
        .when(
            ~F.col("payment_type").isin(1, 2, 3, 4, 5),
            "Excluded: other payment type",
        )
        .otherwise(
            "Included in business dataset"
        ),
    )
    .groupBy("business_status")
    .agg(
        F.count("*").alias("trip_count")
    )
    .withColumn(
        "share_pct",
        F.round(
            F.col("trip_count")
            / F.lit(cleaned_count)
            * 100,
            2,
        ),
    )
    .orderBy(F.desc("trip_count"))
)

cleaning_breakdown.show(
    truncate=False
)

+----------------------------+----------+---------+
|business_status             |trip_count|share_pct|
+----------------------------+----------+---------+
|Included in business dataset|17895220  |76.61    |
|Excluded: payment_type 0    |5417852   |23.19    |
|Excluded: negative fare     |44341     |0.19     |
|Excluded: negative total    |2444      |0.01     |
+----------------------------+----------+---------+



### Business Cleaning Result

After applying the business-specific rules, **17.90 million trips (76.61%)**
remain in the standard business dataset.

The majority of excluded observations are associated with `payment_type = 0`
(**5.42 million trips; 23.19%**). These records are not considered invalid.
Instead, they are retained separately because the preceding analysis identified
a distinct fare structure that is not directly comparable with the standard
fare records used for the initial revenue analysis.

Only a small additional share is excluded because of negative monetary values:

- **44,341 trips (0.19%)** due to negative `fare_amount`
- **2,444 trips (0.01%)** due to negative `total_amount`

This distinction is important: the business dataset is designed specifically
for consistent standard-fare revenue analysis and does not imply that all
excluded records are erroneous.

In [38]:
business_trips.select(
    "fare_amount",
    "total_amount",
    "tip_amount",
    "trip_distance",
).summary(
    "count",
    "min",
    "mean",
    "max",
).show(truncate=False)

+-------+------------------+------------------+------------------+------------------+
|summary|fare_amount       |total_amount      |tip_amount        |trip_distance     |
+-------+------------------+------------------+------------------+------------------+
|count  |17895220          |17895220          |17895220          |17895220          |
|min    |0.0               |0.0               |0.0               |0.0               |
|mean   |19.096408987429893|28.887335155416235|3.7372846614897868|3.5164416732511508|
|max    |863372.12         |863380.37         |960.94            |69735.19          |
+-------+------------------+------------------+------------------+------------------+



In [39]:
for column in [
    "fare_amount",
    "total_amount",
    "tip_amount",
    "trip_distance",
]:
    quantiles = business_trips.approxQuantile(
        column,
        [0.99, 0.995, 0.999, 0.9999],
        0.00001,
    )

    print(f"\n--- {column} ---")

    for p, value in zip(
        [0.99, 0.995, 0.999, 0.9999],
        quantiles,
    ):
        print(
            f"{p * 100:>6.2f}%: {value:,.2f}"
        )


--- fare_amount ---
 99.00%: 80.70
 99.50%: 92.60
 99.90%: 150.00
 99.99%: 315.00

--- total_amount ---
 99.00%: 105.78
 99.50%: 119.47
 99.90%: 184.75
 99.99%: 364.92

--- tip_amount ---
 99.00%: 18.20
 99.50%: 20.85
 99.90%: 30.45
 99.99%: 70.00

--- trip_distance ---
 99.00%: 19.80
 99.50%: 21.55
 99.90%: 29.70
 99.99%: 55.40


### Visual Inspection of Outliers

The upper-tail quantiles indicate that the vast majority of observations remain
within plausible ranges, while the maximum values are several orders of
magnitude larger.

Boxplots are used as a visual diagnostic to examine the distributions and
identify the extent of the upper tails. Sampling is used only for visualization;
the quantitative outlier assessment is based on the complete Spark dataset.

In [41]:
boxplot_sample = (
    business_trips
    .select(
        "fare_amount",
        "total_amount",
        "tip_amount",
        "trip_distance",
    )
    .sample(
        withReplacement=False,
        fraction=0.01,
        seed=42,
    )
    .toPandas()
)

print(f"Sample size: {len(boxplot_sample):,}")

Sample size: 179,666


In [42]:
fig = px.box(
    boxplot_sample,
    y="fare_amount",
    points="outliers",
    title="Fare Amount Distribution",
    labels={
        "fare_amount": "Fare Amount ($)",
    },
)

fig.show()

In [43]:
fig = px.box(
    boxplot_sample[
        boxplot_sample["fare_amount"] <= 315
    ],
    y="fare_amount",
    points="outliers",
    title="Fare Amount Distribution — Up to 99.99th Percentile",
    labels={
        "fare_amount": "Fare Amount ($)",
    },
)

fig.show()

In [44]:
extreme_fares = (
    business_trips
    .filter(
        F.col("fare_amount") > 315
    )
)

print(
    "Extreme fare records:",
    f"{extreme_fares.count():,}"
)

extreme_fares.select(
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "trip_distance",
    "fare_amount",
    "total_amount",
    "tip_amount",
    "tolls_amount",
    "RatecodeID",
    "payment_type",
).orderBy(
    F.desc("fare_amount")
).show(
    30,
    truncate=False,
)

Extreme fare records: 1,847
+--------------------+---------------------+------------+------------+-------------+-----------+------------+----------+------------+----------+------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|PULocationID|DOLocationID|trip_distance|fare_amount|total_amount|tip_amount|tolls_amount|RatecodeID|payment_type|
+--------------------+---------------------+------------+------------+-------------+-----------+------------+----------+------------+----------+------------+
|2025-01-20 12:07:18 |2025-01-20 12:12:42  |138         |8           |1.6          |863372.12  |863380.37   |0.0       |0.0         |1         |4           |
|2025-06-11 14:41:03 |2025-06-11 16:14:37  |161         |132         |20.6         |325478.05  |325528.45   |0.0       |6.94        |2         |2           |
|2025-02-21 17:28:43 |2025-02-21 17:53:04  |161         |246         |2.2          |132531.36  |132555.41   |0.0       |0.0         |1         |3           |
|2025-03-09 00:34:23 |20

### Multivariate Validation of Extreme Fares

The upper-tail analysis shows that high fare amounts cannot be classified as
invalid based on price alone.

Some extreme fares are associated with very long trips and may represent
legitimate long-distance journeys, while others show implausible combinations
of fare, distance, and trip duration.

The following analysis therefore evaluates extreme records using multiple trip
characteristics rather than applying a single fare threshold.

In [45]:
extreme_fares_analysis = (
    extreme_fares
    .withColumn(
        "duration_minutes",
        (
            F.unix_timestamp("tpep_dropoff_datetime")
            - F.unix_timestamp("tpep_pickup_datetime")
        ) / 60,
    )
    .withColumn(
        "fare_per_mile",
        F.when(
            F.col("trip_distance") > 0,
            F.col("fare_amount")
            / F.col("trip_distance"),
        ),
    )
    .withColumn(
        "speed_mph",
        F.when(
            F.col("duration_minutes") > 0,
            F.col("trip_distance")
            / (F.col("duration_minutes") / 60),
        ),
    )
)

In [46]:
(
    extreme_fares_analysis
    .select(
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "trip_distance",
        "duration_minutes",
        "speed_mph",
        "fare_amount",
        "fare_per_mile",
        "total_amount",
        "RatecodeID",
        "payment_type",
    )
    .orderBy(
        F.desc("fare_amount")
    )
    .show(
        20,
        truncate=False,
    )
)

+--------------------+---------------------+-------------+-------------------+------------------+-----------+------------------+------------+----------+------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|duration_minutes   |speed_mph         |fare_amount|fare_per_mile     |total_amount|RatecodeID|payment_type|
+--------------------+---------------------+-------------+-------------------+------------------+-----------+------------------+------------+----------+------------+
|2025-01-20 12:07:18 |2025-01-20 12:12:42  |1.6          |5.4                |17.777777777777775|863372.12  |539607.575        |863380.37   |1         |4           |
|2025-06-11 14:41:03 |2025-06-11 16:14:37  |20.6         |93.56666666666666  |13.20983256145351 |325478.05  |15799.905339805824|325528.45   |2         |2           |
|2025-02-21 17:28:43 |2025-02-21 17:53:04  |2.2          |24.35              |5.42094455852156  |132531.36  |60241.52727272726 |132555.41   |1         |3           |
|202

In [47]:
business_validation = (
    business_trips
    .withColumn(
        "duration_minutes",
        (
            F.unix_timestamp("tpep_dropoff_datetime")
            - F.unix_timestamp("tpep_pickup_datetime")
        ) / 60,
    )
    .withColumn(
        "fare_per_mile",
        F.when(
            F.col("trip_distance") > 0,
            F.col("fare_amount") / F.col("trip_distance"),
        ),
    )
    .withColumn(
        "speed_mph",
        F.when(
            F.col("duration_minutes") > 0,
            F.col("trip_distance")
            / (F.col("duration_minutes") / 60),
        ),
    )
)

In [48]:
validation_columns = [
    "duration_minutes",
    "speed_mph",
    "fare_per_mile",
]

probabilities = [
    0.50,
    0.90,
    0.95,
    0.99,
    0.999,
    0.9999,
]

for column in validation_columns:
    valid_values = (
        business_validation
        .filter(
            F.col(column).isNotNull()
            & (F.col(column) >= 0)
        )
    )

    quantiles = valid_values.approxQuantile(
        column,
        probabilities,
        0.00001,
    )

    print(f"\n--- {column} ---")

    for probability, value in zip(
        probabilities,
        quantiles,
    ):
        print(
            f"{probability * 100:>6.2f}%: "
            f"{value:,.2f}"
        )


--- duration_minutes ---
 50.00%: 11.98
 90.00%: 31.08
 95.00%: 42.62
 99.00%: 70.28
 99.90%: 114.05
 99.99%: 1,432.78

--- speed_mph ---
 50.00%: 9.28
 90.00%: 18.58
 95.00%: 23.92
 99.00%: 34.40
 99.90%: 46.27
 99.99%: 600.00

--- fare_per_mile ---
 50.00%: 7.32
 90.00%: 11.82
 95.00%: 14.13
 99.00%: 22.19
 99.90%: 870.00
 99.99%: 7,500.00


In [49]:
business_validation.select(
    F.sum(
        (F.col("duration_minutes") == 0).cast("long")
    ).alias("zero_duration"),

    F.sum(
        (F.col("speed_mph") > 100).cast("long")
    ).alias("speed_over_100"),

    F.sum(
        (F.col("fare_per_mile") > 100).cast("long")
    ).alias("fare_per_mile_over_100"),
).show()

+-------------+--------------+----------------------+
|zero_duration|speed_over_100|fare_per_mile_over_100|
+-------------+--------------+----------------------+
|       195681|          3919|                 37733|
+-------------+--------------+----------------------+



In [50]:
final_business_trips = (
    business_validation
    .filter(
        (
            F.col("speed_mph").isNull()
            | (F.col("speed_mph") <= 100)
        )
        &
        (
            F.col("fare_per_mile").isNull()
            | (F.col("fare_per_mile") <= 100)
        )
    )
)

In [51]:
before_outlier_cleaning = business_validation.count()
after_outlier_cleaning = final_business_trips.count()

removed_outliers = (
    before_outlier_cleaning
    - after_outlier_cleaning
)

removed_share = (
    removed_outliers
    / before_outlier_cleaning
    * 100
)

print(
    f"Before outlier cleaning: {before_outlier_cleaning:,}"
)
print(
    f"Final business trips:    {after_outlier_cleaning:,}"
)
print(
    f"Removed outliers:        {removed_outliers:,}"
)
print(
    f"Removed share:           {removed_share:.4f}%"
)

Before outlier cleaning: 17,895,220
Final business trips:    17,854,053
Removed outliers:        41,167
Removed share:           0.2300%


### Final Business Dataset

After business-specific filtering and multivariate outlier validation, the final
dataset contains **17,854,053 trips**.

The final outlier validation removed **41,167 records (0.23%)** from the
business dataset. Rather than removing all statistically unusual high-value
trips, the validation focused on clearly implausible combinations of trip
speed and fare per mile.

This preserves legitimate long-distance and high-value trips while reducing
the influence of obvious data-quality errors.

The resulting dataset is used for the revenue and business analysis.

In [52]:
spark.stop()